# WebGPU Preview — SVG vs WebGPU side-by-side

Every converted component renders the **same polars-computed view** through two serializations:
the existing SVG string (`_repr_svg_`) and a WebGPU payload (`.webgpu()` → buffers + manifest,
rendered by the shared runtime in `p2s_webgpu_runtime.py`).

Each case below shows SVG on the left and the WebGPU canvas on the right.
WebGPU requires a browser with `navigator.gpu` (Chrome 113+, Edge, recent Safari/Firefox previews).

In [ ]:
import random
import polars as pl
from IPython.display import HTML, display

import polars2svg
p2s = polars2svg.Polars2SVG()

def side_by_side(component, title=''):
    svg  = component._repr_svg_()
    gpu  = p2s.webgpuHTML(component)
    display(HTML(f'''
      <div style="display:flex;gap:16px;align-items:flex-start;">
        <div><div style="font-family:monospace;font-size:11px;">{title} — SVG</div>
             <div style="border:1px solid #ccc;display:inline-block;">{svg}</div></div>
        <div><div style="font-family:monospace;font-size:11px;">{title} — WebGPU</div>{gpu}</div>
      </div>'''))

random.seed(20260612)
df = pl.DataFrame({
    'x':   [random.random() * 100 for _ in range(2000)],
    'y':   [random.gauss(50, 18) for _ in range(2000)],
    'cat': [random.choice(['alpha', 'beta', 'gamma', 'delta']) for _ in range(2000)],
    'val': [random.random() * 10 for _ in range(2000)],
})

## Histop

In [ ]:
side_by_side(p2s.histop(df, 'cat'), 'histop simple')

In [ ]:
side_by_side(p2s.histop(df, 'cat', color='cat', style=p2s.STACKEDBARp), 'histop stacked')

In [ ]:
side_by_side(p2s.histop(df, 'cat', count='val', style=p2s.BOXPLOTp), 'histop boxplot')

## XYp

In [ ]:
side_by_side(p2s.xyp(df, x='x', y='y', dot_size=2.0), 'xyp circle dots')

In [ ]:
side_by_side(p2s.xyp(df, x='x', y='y', dot_size=2), 'xyp rect dots')

In [ ]:
side_by_side(p2s.xyp(df, x='x', y='y', dot_size=2.0, color=p2s.CROW_MAGNITUDEp), 'xyp crow magnitude')

In [ ]:
side_by_side(p2s.xyp(df, x='x', y='y', dot_size=2.0,
                      x_distributions=p2s.ROW_COUNTp, y_distributions=p2s.ROW_COUNTp), 'xyp distributions')

In [ ]:
side_by_side(p2s.xyp(df.sort('x'), x='x', y='y', line='cat'), 'xyp lines')

In [ ]:
bg = {'zone': [(10, 10), (60, 10), (60, 60), (10, 60)]}
side_by_side(p2s.xyp(df, x='x', y='y', dot_size=2.0,
                     background=bg, background_fill='vary', background_label_color='vary'),
             'xyp background shapes')

## Dense scatter — where WebGPU matters

100K points aggregate to tens of thousands of dot instances; the SVG DOM path is the
bottleneck the GPU path removes.

In [ ]:
import time
t0 = time.time()
big = pl.DataFrame({'x': [random.gauss(50, 15) for _ in range(1_000_000)],
                    'y': [random.gauss(50, 15) for _ in range(1_000_000)]})
t1 = time.time()
print(f'Dataframe Construction Time {t1-t0:0.2f} seconds')
xy_big = p2s.xyp(big, x='x', y='y', dot_size=1.0, wxh=(512, 512), color=p2s.CROW_MAGNITUDEp)
t0 = time.time(); payload = xy_big.webgpu(); t1 = time.time()
n_dots = sum(m['count'] for m in payload['manifest'] if m['kind'] == 'circle')
print(f'{n_dots} dot instances; payload build {1000*(t1-t0):.1f} ms; '
      f'transport {sum(len(v) for v in payload["buffers"].values())/1e6:.1f} MB (base64)')
display(HTML(p2s.webgpuHTML(xy_big)))

## Panel interactive view (`use_webgpu=True`)

The interaction SVG (brush indicator, drag rect, info text, mouse/key capture) sits
transparently above the GPU canvas; brushing and stack ops work exactly as in SVG mode.
Falls back to SVG automatically when the browser has no WebGPU.

In [ ]:
import panel as pn
pn.extension()
from polars2svg.interactive_controller import xypi, histopi, InteractionController

mvc = InteractionController()
mvc.addStack('default', df)
v_xy = xypi(p2s.xyp(df, x='x', y='y', dot_size=2.0), mvc=mvc, use_webgpu=True)
v_h  = histopi(p2s.histop(df, 'cat'), mvc=mvc, use_webgpu=True)
mvc.view_stack[id(v_xy)] = 'default'; mvc.view_refs[id(v_xy)] = v_xy
mvc.view_stack[id(v_h)]  = 'default'; mvc.view_refs[id(v_h)]  = v_h
pn.Row(v_xy, v_h)

In [ ]:
v_xy = p2s.xyp(df, x='x', y='y', dot_size=2.0, color='cat')
v_h  = p2s.histop(df, 'cat', color='cat')
p2s.panelize([[v_xy, v_h]], use_webgpu=True)

## Phases 2–5 — Timep, LinkP, ChP, SpreadLinesP, Smallp

Every component now has both representations. GPU approximations to note:
LinkP collapsed-node *clouds* render as circles; ChP circular labels are placed
per-glyph along the arc; SpreadLinesP recovers primitives by parsing its SVG
(generic `svgToDisplayList` fallback).

In [ ]:
import datetime
random.seed(20260612)
nodes = [f'n{i}' for i in range(15)]
df2 = pl.DataFrame({
    'ts':  [datetime.datetime(2026, 1, 1 + i % 28, i % 24) for i in range(600)],
    'cat': [random.choice('abcd') for _ in range(600)],
    'fm':  [random.choice(nodes) for _ in range(600)],
    'to':  [random.choice(nodes) for _ in range(600)],
    'val': [random.random() for _ in range(600)],
})
pos = {n: (random.random(), random.random()) for n in nodes}

### Timep

In [ ]:
side_by_side(p2s.timep(df2, 'ts'), 'timep simple')

In [ ]:
side_by_side(p2s.timep(df2, 'ts', style=p2s.STACKEDBARp, color='cat'), 'timep stacked')

### LinkP

In [ ]:
side_by_side(p2s.linkp(df=df2, relationships=[('fm','to')], pos=pos), 'linkp line links')

In [ ]:
side_by_side(p2s.linkp(df=df2, relationships=[('fm','to')], pos=pos,
                       link_shape='curve', link_size='vary', node_size='vary', draw_labels=True),
             'linkp curves + vary + labels')

### ChP (chord diagram)

In [ ]:
side_by_side(p2s.chordp(df=df2, relationships=[('fm','to')], wxh=(256,256), draw_labels=True), 'chordp radial labels')

In [ ]:
side_by_side(p2s.chordp(df=df2, relationships=[('fm','to')], wxh=(256,256),
                        draw_labels=True, label_style='circular'), 'chordp circular labels')

In [ ]:
side_by_side(p2s.chordp(df=df2, relationships=[('fm','to')], wxh=(256,256), link_shape='bundled'), 'chordp bundled')

### SpreadLinesP

In [ ]:
side_by_side(p2s.spreadlinesp(df2, [('fm','to')], ego='n0', time='ts'), 'spreadlinesp')

### Smallp (small multiples — cells composed from each component's display list)

In [ ]:
tmpl = p2s.histop(df2, 'cat', wxh=(96,96))
side_by_side(p2s.smallp(df2, 'fm', tmpl, wxh=(420,280)), 'smallp of histop')

### XYp gradient lines (Phase 5 — per-vertex-colored quads on GPU)

In [ ]:
df_l = pl.DataFrame({'x': sorted(random.random() for _ in range(80)),
                     'y': [random.random() for _ in range(80)],
                     'sample': [random.choice('ab') for _ in range(80)],
                     'value': [random.random() for _ in range(80)]})
side_by_side(p2s.xyp(df_l, x='x', y='y', color='value', dot_size='value', opacity='value',
                     line=('sample', p2s.LINECOLOR_FIELD)), 'xyp gradient lines')

### Full-board interactive (`panelize(..., use_webgpu=True)`)

All components in one linked board — every view renders on its own GPU canvas.

In [ ]:
board = p2s.panelize([[p2s.xyp(df2, x='val', y='val', dot_size=2.0),
                       p2s.histop(df2, 'cat'),
                       p2s.linkp(df2, [('fm','to')])],
                      [p2s.timep(df2, 'ts'),
                       p2s.chordp(df=df2, relationships=[('fm','to')], wxh=(256,256))]],
                     use_webgpu=True)
board

## Interactive GPU views — linkpi & spreadlinepi

`linkpi(..., use_webgpu=True)` and `spreadlinepi(..., use_webgpu=True)` render the
plot on a `<canvas>` while interaction chrome (selection highlight, hit-test layer,
drag rect, layout-helper shapes, info/search text) stays as an SVG overlay. Drag a
node / run a layout op (`w`) / zoom — the canvas re-renders; selection + hit-testing
still work. Falls back to SVG when `navigator.gpu` is absent.

In [ ]:
from polars2svg.interactive_controller import linkpi
from polars2svg.spreadlinepi import spreadlinepi
random.seed(20260612)
gnodes = [f'n{i}' for i in range(18)]
gdf = pl.DataFrame({'fm':[random.choice(gnodes) for _ in range(500)],
                    'to':[random.choice(gnodes) for _ in range(500)],
                    'ts':[datetime.datetime(2026,1,1+i%20, i%24) for i in range(500)]})
gpos = {n:(random.random(), random.random()) for n in gnodes}

### linkpi — interactive network on GPU

In [ ]:
linkpi(p2s.linkp(df=gdf, relationships=[('fm','to')], pos=gpos, draw_labels=True), use_webgpu=True)

### spreadlinepi — interactive egocentric spread on GPU

In [ ]:
spreadlinepi(p2s.spreadlinesp(gdf, [('fm','to')], ego='n0', time='ts'), use_webgpu=True)

### Combined linked board (every interactive component on GPU)

In [ ]:
p2s.panelize([[p2s.linkp(df=gdf, relationships=[('fm','to')], pos=gpos),
               p2s.spreadlinesp(gdf, [('fm','to')], ego='n0', time='ts')]],
             use_webgpu=True)